# 🚀 Medical VLM Decisive Validation — Qwen2.5-VL-7B (rev05b End-to-End)

**Research Project:** Selective Prediction for Bilingual Medical Vision-Language Models Under Language Shift (IndoRad-VQA)  
**Target Model:** `Qwen/Qwen2.5-VL-7B-Instruct` (Qwen2.5-VL-7B)  
**Execution Mode:** End-to-End (Inference / Checkpoint-Resume + Full Statistical Selective Prediction Pipeline)  
**Revision:** `rev05b` (Modular & Fault-Tolerant)  

---

### 📌 Notebook Architecture & Scientific Capabilities:
1. **Dedicated Single-Model Run:** Evaluates `Qwen2.5-VL-7B` in isolation (eliminates dual-model timeout risk).
2. **Resilient Set-Based Resuming:** If a completed/partial checkpoint is attached via `/kaggle/input`, it instantly resumes or skips inferencing (100% Zero Timeout).
3. **Dual-Track Evaluation:** Computes both Sequence Log-Likelihood and Length-Normalized Surprisal for EN (Yes/No) and ID (Ya/Tidak).
4. **Visual Intervention (P0 vs P1):** Measures directional sensitivity against pure black canvas (0,0,0).
5. **Selective Prediction (RQ3):** Computes Discrete Riemann AURC across all 5 estimators (M1, M2, M_dir, M_norm, M3_old) with Cluster Bootstrap (B=2,000) on 204 patient image clusters.
6. **Inline Risk-Coverage Plot:** Automatically displays the high-resolution Risk-Coverage curve directly in the notebook cell.
7. **Learned Risk Ablation:** Trains Logistic Regression risk models on Dev (N=323) with GroupKFold and tests incremental value of visual sensitivity and cross-lingual disagreement (D_lang) on Primary Test (N=799).


In [ ]:
# ==============================================================================
# STAGE 1: ENVIRONMENT & HARDWARE AUDIT
# ==============================================================================
import os
import sys
import subprocess
import platform

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

try:
    from IPython.display import display, FileLink
except ImportError:
    display = print
    FileLink = str

print("=" * 70)
print("🔍 SYSTEM & HARDWARE AUDIT")
print("=" * 70)
print(f"OS Platform     : {platform.platform()}")
print(f"Python Version  : {sys.version.split()[0]}")

required_pkgs = {
    "accelerate": "accelerate>=0.26.0",
    "bitsandbytes": "bitsandbytes",
    "qwen_vl_utils": "qwen-vl-utils",
    "huggingface_hub": "huggingface_hub",
    "pyarrow": "pyarrow",
    "pandas": "pandas",
    "numpy": "numpy"
}

pkgs_to_install = []
for mod_name, pip_name in required_pkgs.items():
    try:
        __import__(mod_name)
    except ImportError:
        pkgs_to_install.append(pip_name)

# Ensure transformers >= 4.49.0
try:
    import transformers
    if not hasattr(transformers, "Qwen2_5_VLForConditionalGeneration"):
        pkgs_to_install.append("transformers>=4.49.0")
except Exception:
    pkgs_to_install.append("transformers>=4.49.0")

if pkgs_to_install:
    unique_pkgs = sorted(list(set(pkgs_to_install)))
    print(f"📦 Installing/Upgrading packages: {unique_pkgs}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + unique_pkgs)
    import importlib
    importlib.invalidate_caches()
    if "transformers" in sys.modules:
        importlib.reload(sys.modules["transformers"])
    print("✅ Packages successfully installed.")

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"Device Count    : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"  - GPU {i}: {props.name} ({vram_gb:.2f} GB VRAM, CC {props.major}.{props.minor})")
else:
    print("⚠️ WARNING: No CUDA GPU detected!")
print("=" * 70)

In [ ]:
# ==============================================================================
# STAGE 2: EXPERIMENT CONFIGURATION & RESUME PARAMETERS
# ==============================================================================
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import torch

REVISION_ID = "rev06"
# Hardware precision & parity settings
# STRICT PARITY RULE: rev05 used torch.bfloat16. Default to torch.bfloat16 to prevent
# confounding token-set comparisons with dtype variations and avoid fp16 overflow in Qwen2.5-VL.
import platform
COMPUTE_DTYPE = torch.bfloat16
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor()

# Execution & Pilot Mode:
# When PILOT_MODE = True, executes a 40-sample stratified preflight verification (12 dev, 28 test)
# and compares recomputed scores against rev05 to verify numerical stability before full run.
PILOT_MODE = False
PILOT_SAMPLE_COUNT = 40
SCORING_METHOD = "exact_sequence_candidate_normalized_v5"
AUTHORITATIVE_MANIFEST_SHA256 = "fb55fd75378b1c03abcbeb9772fa27d6ab81dd9abdfca4735c03438507c19ee9"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_SLUG = MODEL_NAME.replace("/", "_").replace("-", "_")

# Batching & execution control
COHORT_MODE = "active_union"   # Evaluates all 1,133 QA pairs (Primary 1,122 + Historical 150)
BATCH_SIZE = 5                 # Flush atomic checkpoint every 5 samples
START_INDEX = 0                # Row offset within remaining samples (default: 0)
MAX_SAMPLES = None             # None = process all remaining samples

# Output directories (isolated for Qwen2.5-VL-7B)
RESULTS_ROOT = Path("results/decisive_validation/rev06")
MODEL_OUT_DIR = RESULTS_ROOT / MODEL_SLUG
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
PERTURB_DIR = RESULTS_ROOT / "perturbed_images"
PERTURB_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_EN_TEMPLATE = "Answer only YES or NO.\nQuestion: {q}"
PROMPT_ID_TEMPLATE = "Jawab hanya dengan YA atau TIDAK.\nPertanyaan: {q}"

CANDIDATES_EN = ["YES<|im_end|>", "NO<|im_end|>"]
CANDIDATES_ID = ["YA<|im_end|>", "TIDAK<|im_end|>"]


# rev06 Unified Token Sets for Exact LogSumExp Posterior Scoring
TOKEN_SETS_EN = {
    "yes": ["YES", " YES", "Yes", " Yes"],
    "no": ["NO", " NO", "No", " No"]
}
TOKEN_SETS_ID = {
    "ya": ["YA", " YA", "Ya", " Ya", "ya", " ya"],
    "tidak": ["TIDAK", " TIDAK", "Tidak", " Tidak", "tidak", " tidak"]
}

LABEL_MAP = {
    "YES<|im_end|>": "YES",
    "NO<|im_end|>": "NO",
    "YA<|im_end|>": "YES",
    "TIDAK<|im_end|>": "NO"
}

EMBEDDED_MANIFEST_B64 = "..."

MODEL_NAME_SHORT = "Qwen2.5-VL-7B"


In [ ]:
# ==============================================================================
# STAGE 3: AUTHORITATIVE FROZEN MANIFEST ENFORCEMENT & COHORT SELECTION
# ==============================================================================
import zlib
import base64
import hashlib
import io
import pandas as pd

def load_and_verify_manifest():
    search_paths = [
        Path("main/results/manifest_rev05.csv"),
        Path("main/results/decisive_validation/rev05/manifest_rev05.csv"),
        Path("results/decisive_validation/rev05/manifest_rev05.csv"),
        Path("manifest_rev05.csv"),
        Path("./manifest_rev05.csv")
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob("**/manifest_rev05.csv"):
            search_paths.append(p)
            
    df_manifest = None
    loaded_source = None
    for p in search_paths:
        if p.exists():
            raw_bytes = p.read_bytes()
            if hashlib.sha256(raw_bytes).hexdigest() == AUTHORITATIVE_MANIFEST_SHA256:
                df_manifest = pd.read_csv(io.BytesIO(raw_bytes))
                loaded_source = str(p)
                break
                
    if df_manifest is None:
        print("ℹ️ Local manifest file not found. Decompressing embedded base64 verified manifest...")
        decompressed_bytes = zlib.decompress(base64.b64decode(EMBEDDED_MANIFEST_B64))
        calc_sha = hashlib.sha256(decompressed_bytes).hexdigest()
        assert calc_sha == AUTHORITATIVE_MANIFEST_SHA256,             f"Embedded manifest hash mismatch! Expected {AUTHORITATIVE_MANIFEST_SHA256}, got {calc_sha}"
        df_manifest = pd.read_csv(io.BytesIO(decompressed_bytes))
        loaded_source = "EMBEDDED_VERIFIED_BASE64"
        
    assert len(df_manifest) == 2244, f"Expected 2244 total QA rows, got {len(df_manifest)}"
    assert df_manifest["canonical_image_id"].nunique() == 314, "Expected 314 canonical images!"
    
    # Active Evaluation Cohort Union (N=1,133)
    df_union = df_manifest[(df_manifest["is_eligible_primary"] == True) | (df_manifest["in_rev03_benchmark"] == True)].copy()
    df_union = df_union.sort_values("qa_id").reset_index(drop=True)
    assert len(df_union) == 1133, f"Expected 1133 union rows, got {len(df_union)}"
    
    print(f"✅ Authoritative Manifest 100% Cryptographically Verified ({loaded_source})")
    print(f"   SHA-256 Manifest         : {AUTHORITATIVE_MANIFEST_SHA256}")
    print(f"   Total IndoRad-VQA Rows   : {len(df_manifest)}")
    print(f"   Active Evaluation Cohort : {len(df_union)} (806 Test / 327 Dev | 0% Leakage)")
    return df_union

df_eval_cohort = load_and_verify_manifest()

In [ ]:
# ==============================================================================
# STAGE 4: IMAGE RESOLUTION & PIXEL SHA-256 VERIFICATION
# ==============================================================================
from PIL import Image
import io
import hashlib
import zipfile

def compute_rgb_pixel_sha256(pil_img):
    rgb = pil_img.convert("RGB")
    return hashlib.sha256(rgb.tobytes()).hexdigest()

def resolve_all_images(df):
    cache_dir = RESULTS_ROOT / "verified_images"
    cache_dir.mkdir(parents=True, exist_ok=True)
    
    search_dirs = [
        cache_dir,
        Path("main/data/images"),
        Path("data/images"),
        Path("./images")
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for zf in kaggle_input.rglob("*.zip"):
            try:
                print(f"📦 Auto-extracting zip archive: {zf.name} to {cache_dir}...")
                with zipfile.ZipFile(zf, "r") as z:
                    z.extractall(cache_dir)
            except Exception as e:
                print(f"⚠️ Warning: Could not extract {zf.name}: {e}")
                
        for p in kaggle_input.rglob("*.png"):
            if p.parent not in search_dirs:
                search_dirs.append(p.parent)
                
    resolved_map = {}
    missing_rows = []
    
    unique_hashes = df["pixel_sha256"].unique()
    print(f"🔍 Cryptographically verifying pixel SHA-256 for all {len(unique_hashes)} unique physical images...")
    
    disk_cache = {}
    for sdir in search_dirs:
        for p in sdir.glob("*.png"):
            fname = p.stem.lower()
            if len(fname) == 16:
                disk_cache[fname] = p

    verified_cid_cache = {}
    for _, row in df.iterrows():
        qa_id = row["qa_id"]
        cid = str(row["canonical_image_id"]).strip().lower()
        if cid in verified_cid_cache:
            resolved_map[qa_id] = verified_cid_cache[cid]
            continue

        expected_hash = str(row["pixel_sha256"]).strip().lower()
        expected_w = int(row["image_width"])
        expected_h = int(row["image_height"])
        
        found = None
        if cid in disk_cache:
            p = disk_cache[cid]
            try:
                with Image.open(p) as img:
                    if img.size == (expected_w, expected_h):
                        if compute_rgb_pixel_sha256(img) == expected_hash:
                            found = p
            except Exception:
                pass
                
        if found is None:
            for sdir in search_dirs:
                for cand_name in [f"{cid}.png", f"img_{int(qa_id):05d}.png", f"{qa_id}.png", f"{row.get('image_name', '')}"]:
                    if not cand_name:
                        continue
                    p = sdir / cand_name
                    if p.exists():
                        try:
                            with Image.open(p) as img:
                                if img.size == (expected_w, expected_h):
                                    if compute_rgb_pixel_sha256(img) == expected_hash:
                                        found = p
                                        break
                        except Exception:
                            pass
                if found is not None:
                    break
                
        if found is not None:
            resolved_map[qa_id] = found
            verified_cid_cache[cid] = found
        else:
            missing_rows.append(row)
            
    if missing_rows:
        print(f"⚠️ {len(missing_rows)} images missing from local search dirs. Downloading via Parquet...")
        import pyarrow.parquet as pq
        from huggingface_hub import hf_hub_download
        splits_needed = set()
        for r in missing_rows:
            splits_needed.add("test" if "test" in str(r["split"]).lower() else "train")
            
        for sname in splits_needed:
            fname = f"data/{sname}-00000-of-00001.parquet"
            local_pq = hf_hub_download(repo_id="Lab-IS/IndoRad-VQA", filename=fname, repo_type="dataset")
            table = pq.read_table(local_pq)
            df_pq = table.to_pandas()
            for r in missing_rows:
                qid = r["qa_id"]
                cid = str(r["canonical_image_id"])
                exp_h = str(r["pixel_sha256"]).strip().lower()
                match = df_pq[df_pq["qid"].astype(str) == str(qid)]
                if len(match) == 0:
                    continue
                img_val = match.iloc[0]["image"]
                pil_img = None
                try:
                    if isinstance(img_val, dict) and "bytes" in img_val:
                        pil_img = Image.open(io.BytesIO(img_val["bytes"]))
                    elif hasattr(img_val, "convert"):
                        pil_img = img_val
                    elif isinstance(img_val, bytes):
                        pil_img = Image.open(io.BytesIO(img_val))
                except Exception:
                    pass
                if pil_img and compute_rgb_pixel_sha256(pil_img) == exp_h:
                    sp = cache_dir / f"{cid}.png"
                    pil_img.save(sp)
                    resolved_map[qid] = sp
                            
    unresolved = [r["qa_id"] for _, r in df.iterrows() if r["qa_id"] not in resolved_map]
    assert len(unresolved) == 0, f"Critical failure: {len(unresolved)} images unresolved!"
    print(f"✅ Cryptographic Image Verification: 100% PASS ({len(resolved_map)} images verified).")
    return resolved_map

image_map = resolve_all_images(df_eval_cohort)

In [ ]:
# ==============================================================================
# STAGE 5: EXACT SEQUENCE LOG-LIKELIHOOD & KV-CACHE MULTI-TOKEN SCORING ENGINE
# ==============================================================================
import torch
import torch.nn.functional as F
from qwen_vl_utils import process_vision_info

def extract_target_loglikelihood_v5(model, processor, image_path, prompt_text, candidate_strings, target_device):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path), "min_pixels": 256 * 28 * 28, "max_pixels": 512 * 512},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    prompt_str = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    cand_stats = []
    seq_logprobs = []
    len_norm_logprobs = []
    
    for cand in candidate_strings:
        full_text = prompt_str + cand
        prompt_inputs = processor(text=[prompt_str], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
        full_inputs = processor(text=[full_text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
        
        prompt_len = prompt_inputs["input_ids"].shape[1]
        target_ids = full_inputs["input_ids"][:, prompt_len:].to(target_device)
        cand_len = target_ids.shape[1]
        full_inputs = full_inputs.to(target_device)
        
        with torch.no_grad():
            logits = model(**full_inputs).logits
            
        target_logits = logits[:, prompt_len - 1 : -1, :]
        log_probs = F.log_softmax(target_logits, dim=-1)
        token_logprobs = torch.gather(log_probs, 2, target_ids.unsqueeze(-1)).squeeze(-1)
        
        seq_lp = token_logprobs.sum().item()
        len_lp = seq_lp / max(1, cand_len)
        
        seq_logprobs.append(seq_lp)
        len_norm_logprobs.append(len_lp)
        cand_stats.append({"cand": cand, "target_ids": target_ids[0].tolist(), "cand_len": cand_len, "seq_lp": seq_lp, "len_lp": len_lp})
        
    s_aff_seq, s_neg_seq = seq_logprobs[0], seq_logprobs[1]
    max_seq = max(s_aff_seq, s_neg_seq)
    p_aff_seq = float(np.exp(s_aff_seq - max_seq) / (np.exp(s_aff_seq - max_seq) + np.exp(s_neg_seq - max_seq)))
    p_neg_seq = float(1.0 - p_aff_seq)
    
    s_aff_len, s_neg_len = len_norm_logprobs[0], len_norm_logprobs[1]
    max_len = max(s_aff_len, s_neg_len)
    p_aff_len = float(np.exp(s_aff_len - max_len) / (np.exp(s_aff_len - max_len) + np.exp(s_neg_len - max_len)))
    p_neg_len = float(1.0 - p_aff_len)
    
    return cand_stats, (p_aff_seq, p_neg_seq), (p_aff_len, p_neg_len)

def score_binary_candidates_v5(model, processor, image_path, prompt_text, candidate_strings, target_device):
    cand_stats, (p_aff_seq, p_neg_seq), (p_aff_len, p_neg_len) = extract_target_loglikelihood_v5(
        model, processor, image_path, prompt_text, candidate_strings, target_device
    )
    c_aff, c_neg = candidate_strings[0], candidate_strings[1]
    lbl_aff = LABEL_MAP[c_aff]
    lbl_neg = LABEL_MAP[c_neg]
    pred_seq = lbl_aff if p_aff_seq >= 0.5 else lbl_neg
    conf_seq = p_aff_seq if p_aff_seq >= 0.5 else p_neg_seq
    pred_len = lbl_aff if p_aff_len >= 0.5 else lbl_neg
    conf_len = p_aff_len if p_aff_len >= 0.5 else p_neg_len
    return {
        "pred_seq": pred_seq, "conf_seq": conf_seq, "p_aff_seq": p_aff_seq, "p_neg_seq": p_neg_seq,
        "pred_len": pred_len, "conf_len": conf_len, "p_aff_len": p_aff_len, "p_neg_len": p_neg_len,
        "cand_stats": cand_stats
    }

def score_condition_with_kv_cache(model, processor, image_path, prompt_text, candidates_old, token_sets_v6, target_device, use_cache=True):
    """
    Evaluates both rev05 candidate strings (YA/TIDAK or YES/NO<|im_end|>)
    AND rev06 LogSumExp token sets in EXACTLY ONE multimodal prefix forward pass using KV-caching.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path), "min_pixels": 256 * 28 * 28, "max_pixels": 512 * 512},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    prompt_str = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    prompt_inputs = processor(
        text=[prompt_str],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(target_device)
    
    with torch.no_grad():
        outputs = model(**prompt_inputs, use_cache=True)
        past_kv = outputs.past_key_values if use_cache else None
        next_token_logits = outputs.logits[0, -1, :]
        log_probs_prefix = F.log_softmax(next_token_logits, dim=-1)
        
    seq_len = prompt_inputs["input_ids"].shape[1]
    tok = processor.tokenizer
    
    def score_single_string(s):
        t_ids = tok.encode(s, add_special_tokens=False)
        if len(t_ids) == 0:
            return -999.0
        lp = log_probs_prefix[t_ids[0]].item()
        if len(t_ids) == 1:
            return lp
        sub_lp = lp
        curr_kv = past_kv
        for step in range(len(t_ids) - 1):
            inp_step = torch.tensor([[t_ids[step]]], device=target_device)
            cache_pos = torch.tensor([seq_len + step], device=target_device)
            next_pos_ids = torch.tensor([seq_len + step], device=target_device).view(1, 1, 1).expand(3, 1, 1)
            with torch.no_grad():
                step_out = model(
                    input_ids=inp_step,
                    position_ids=next_pos_ids,
                    past_key_values=curr_kv,
                    use_cache=True,
                    cache_position=cache_pos
                )
                curr_kv = step_out.past_key_values
                step_lps = F.log_softmax(step_out.logits[0, -1, :], dim=-1)
                sub_lp += step_lps[t_ids[step + 1]].item()
        return sub_lp

    # 1. Recompute rev05 candidates via KV-cache
    c_aff, c_neg = candidates_old[0], candidates_old[1]
    s_aff = score_single_string(c_aff)
    s_neg = score_single_string(c_neg)
    max_s = max(s_aff, s_neg)
    p_aff_seq = float(np.exp(s_aff - max_s) / (np.exp(s_aff - max_s) + np.exp(s_neg - max_s)))
    p_neg_seq = float(1.0 - p_aff_seq)
    lbl_aff = LABEL_MAP[c_aff]
    lbl_neg = LABEL_MAP[c_neg]
    pred_seq = lbl_aff if p_aff_seq >= 0.5 else lbl_neg
    conf_seq = p_aff_seq if p_aff_seq >= 0.5 else p_neg_seq

    len_aff = max(1, len(tok.encode(c_aff, add_special_tokens=False)))
    len_neg = max(1, len(tok.encode(c_neg, add_special_tokens=False)))
    s_aff_len = s_aff / len_aff
    s_neg_len = s_neg / len_neg
    max_sl = max(s_aff_len, s_neg_len)
    p_aff_len = float(np.exp(s_aff_len - max_sl) / (np.exp(s_aff_len - max_sl) + np.exp(s_neg_len - max_sl)))
    p_neg_len = float(1.0 - p_aff_len)
    pred_len = lbl_aff if p_aff_len >= 0.5 else lbl_neg
    conf_len = p_aff_len if p_aff_len >= 0.5 else p_neg_len

    # 2. Evaluate rev06 multi-token LogSumExp sets
    class_logprobs = {}
    for class_name, cand_list in token_sets_v6.items():
        v_lps = [score_single_string(c_str) for c_str in cand_list]
        v_lps = [lp for lp in v_lps if lp > -900.0]
        if len(v_lps) > 0:
            class_logprobs[class_name] = float(torch.logsumexp(torch.tensor(v_lps), dim=0).item())
        else:
            class_logprobs[class_name] = -999.0
            
    classes = list(token_sets_v6.keys())
    c0, c1 = classes[0], classes[1]
    s0, s1 = class_logprobs[c0], class_logprobs[c1]
    max_v6 = max(s0, s1)
    p_aff_v6 = float(np.exp(s0 - max_v6) / (np.exp(s0 - max_v6) + np.exp(s1 - max_v6)))
    p_neg_v6 = float(1.0 - p_aff_v6)
    pred_v6 = "YES" if p_aff_v6 >= 0.5 else "NO"
    conf_v6 = p_aff_v6 if p_aff_v6 >= 0.5 else p_neg_v6

    return {
        "pred_seq": pred_seq, "conf_seq": conf_seq, "p_aff_seq": p_aff_seq, "p_neg_seq": p_neg_seq,
        "pred_len": pred_len, "conf_len": conf_len, "p_aff_len": p_aff_len, "p_neg_len": p_neg_len,
        "pred_v6": pred_v6, "conf_v6": conf_v6, "p_aff_v6": p_aff_v6, "p_neg_v6": p_neg_v6,
        "s_aff": s_aff, "s_neg": s_neg, "class_logprobs": class_logprobs
    }

print("✅ Scoring Engine (rev05 Sequence + rev06 Prefix KV-Cache Multi-Token LogSumExp) initialized.")


In [ ]:
# ==============================================================================
# STAGE 6: STRICT CHECKPOINT METADATA AUDIT & SET-BASED RESUME
# ==============================================================================
import shutil

checkpoint_filename = f"live_validation_checkpoint_{MODEL_SLUG}.csv"
checkpoint_dest = MODEL_OUT_DIR / checkpoint_filename

candidate_search_paths = [
    checkpoint_dest,
    Path("/kaggle/working") / checkpoint_filename,
    Path("/kaggle/working/decisive_validation_rev05") / MODEL_SLUG / checkpoint_filename,
    Path("main/results/decisive_validation/rev05") / MODEL_SLUG / checkpoint_filename,
    Path("main/results/results 08_decisive_validation_rev05/decisive_validation/rev05") / MODEL_SLUG / checkpoint_filename
]

kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    for p in kaggle_input.rglob("*.csv"):
        nl = p.name.lower()
        if "qwen" in nl or "checkpoint" in nl or MODEL_SLUG.lower() in nl:
            candidate_search_paths.insert(0, p)

df_existing_ckpt = None
found_ckpt_path = None

for p in candidate_search_paths:
    if p.exists() and p.is_file() and p.stat().st_size > 1000:
        try:
            df_temp = pd.read_csv(p)
            # Metadata compatibility check
            if (len(df_temp) > 0 and 
                df_temp["revision_id"].iloc[0] in [REVISION_ID, "rev05b"] and
                df_temp["manifest_sha256"].iloc[0] == AUTHORITATIVE_MANIFEST_SHA256 and
                df_temp["model_id"].iloc[0] == MODEL_NAME and
                np.isfinite(df_temp["score_m_norm_seq"]).all() and
                df_temp["qa_id"].duplicated().sum() == 0):
                df_existing_ckpt = df_temp
                found_ckpt_path = p
                print(f"🔄 Valid compatible checkpoint found at: {p} ({len(df_existing_ckpt)} rows)")
                break
            else:
                print(f"⚠️ Checkpoint at {p} failed compatibility validation. Skipping...")
        except Exception as e:
            print(f"⚠️ Failed to parse {p}: {e}")

# If loaded from read-only /kaggle/input, copy to working destination
if found_ckpt_path and found_ckpt_path.resolve() != checkpoint_dest.resolve():
    shutil.copy2(found_ckpt_path, checkpoint_dest)
    print(f"📋 Copied existing checkpoint to active working location: {checkpoint_dest}")

completed_qa_ids = set(df_existing_ckpt["qa_id"]) if df_existing_ckpt is not None else set()
print(f"📊 Completed QA IDs in Checkpoint: {len(completed_qa_ids)} / {len(df_eval_cohort)}")

# True Set-Based Resume
df_remaining = df_eval_cohort[~df_eval_cohort["qa_id"].isin(completed_qa_ids)].copy().reset_index(drop=True)

if START_INDEX > 0:
    print(f"⏩ Applying START_INDEX offset: {START_INDEX}")
    df_remaining = df_remaining.iloc[START_INDEX:].reset_index(drop=True)
if MAX_SAMPLES is not None and MAX_SAMPLES > 0:
    print(f"🛑 Capping run to MAX_SAMPLES: {MAX_SAMPLES}")
    df_remaining = df_remaining.iloc[:MAX_SAMPLES].reset_index(drop=True)

print(f"🎯 Remaining Samples to Evaluate: {len(df_remaining)}")
if len(df_remaining) > 0:
    print(f"   First Remaining QA ID        : {df_remaining.iloc[0]['qa_id']}")
    print(f"   Last Remaining QA ID         : {df_remaining.iloc[-1]['qa_id']}")
    print(f"   Split Breakdown of Remaining : {dict(df_remaining['split'].value_counts())}")
    est_duration_min = (len(df_remaining) * 20.93) / 60.0
    print(f"   Estimated Inference Runtime  : {est_duration_min:.2f} minutes (~{est_duration_min/60.0:.2f} hours)")
else:
    print("🎉 All 1,133 samples already complete! Ready for aggregation.")

In [ ]:
# ==============================================================================
# STAGE 7: LOAD QWEN2.5-VL-7B (4-BIT NF4 QUANTIZATION)
# ==============================================================================
import gc
from transformers import AutoProcessor, AutoConfig, BitsAndBytesConfig
import transformers

if len(df_remaining) == 0:
    print("⏭️ Inference complete; skipping model loading.")
    model = None
    processor = None
    target_dev = None
else:
    print(f"🚀 Loading {MODEL_NAME} with 4-bit NF4 Quantization...")
    config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=False)
    arch_list = getattr(config, "architectures", [])
    model_class = None
    for arch_name in arch_list:
        if hasattr(transformers, arch_name):
            model_class = getattr(transformers, arch_name)
            break
    if model_class is None:
        from transformers import AutoModelForVision2Seq
        model_class = AutoModelForVision2Seq

    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=False)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = model_class.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        trust_remote_code=False
    )
    model.eval()
    target_dev = model.device if hasattr(model, "device") else next(model.parameters()).device
    print(f"✅ Model successfully placed on device: {target_dev}")


In [ ]:
# ==============================================================================
# STAGE 8A: INFERENCE QUEUE SETUP & STRATIFIED PILOT SAMPLING
# ==============================================================================
import os
import shutil
import time
import numpy as np
import pandas as pd

def atomic_save(new_records, dest_path, df_base=None):
    df_new = pd.DataFrame(new_records)
    if df_base is not None and len(df_base) > 0:
        df_all = pd.concat([df_base, df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all = df_all.drop_duplicates(subset=["qa_id"], keep="last").reset_index(drop=True)
    tmp_path = dest_path.parent / f"{dest_path.stem}.tmp_{os.getpid()}.csv"
    df_all.to_csv(tmp_path, index=False)
    shutil.move(tmp_path, dest_path)
    return df_all

# Enforce active evaluation cohort dataframe source
cohort_source = df_eval_cohort if "df_eval_cohort" in globals() else (df_manifest if "df_manifest" in globals() else None)
assert cohort_source is not None, "❌ Dataframe kohort evaluasi tidak ditemukan! Jalankan Sel 3 terlebih dahulu."

if PILOT_MODE:
    checkpoint_dest = MODEL_OUT_DIR / f"pilot_checkpoint_{MODEL_SLUG}.csv"
    print(f"🔬 PILOT PREFLIGHT MODE ACTIVE: Stratifying {PILOT_SAMPLE_COUNT} samples (12 Dev + 28 Test, Seed: {RANDOM_SEED})...")
    dev_pool = cohort_source[(cohort_source["split"] == "dev") & (cohort_source["is_eligible_primary"] == True)]
    test_pool = cohort_source[(cohort_source["split"] == "test") & (cohort_source["is_eligible_primary"] == True)]
    n_dev_sample = min(12, len(dev_pool))
    n_test_sample = min(28, len(test_pool))
    pilot_dev = dev_pool.sample(n=n_dev_sample, random_state=RANDOM_SEED)
    pilot_test = test_pool.sample(n=n_test_sample, random_state=RANDOM_SEED)
    df_queue = pd.concat([pilot_dev, pilot_test]).sort_values("qa_id").reset_index(drop=True)
else:
    checkpoint_dest = MODEL_OUT_DIR / f"live_validation_checkpoint_{MODEL_SLUG}.csv"
    df_queue = df_remaining

print(f"📁 Target Checkpoint Path : {checkpoint_dest}")
print(f"🎯 Antrean Sampel Diproses: {len(df_queue)} sampel")
print(f"   Breakdown Split        : {dict(df_queue['split'].value_counts())}\n")


In [ ]:
# ==============================================================================
# STAGE 8B: RESILIENT INFERENCE LOOP (KV-CACHE FORWARD & ATOMIC FLUSH)
# ==============================================================================
from tqdm.auto import tqdm
import time
import torch

new_records = []

if len(df_queue) > 0 and model is not None:
    new_records = []
    t_start = time.time()
    
    print(f"⚡ Starting inference on {len(df_queue)} samples (Batch Size: {BATCH_SIZE}, Compute Dtype: {COMPUTE_DTYPE}, Device: {target_dev})...")
    
    for i, (_, row) in enumerate(tqdm(df_queue.iterrows(), total=len(df_queue), desc=f"Qwen2.5-VL-7B Inference")):
        qa_id = row["qa_id"]
        cid = str(row["canonical_image_id"])
        img_id = str(row.get("image_id", cid))
        split_val = str(row.get("split", "test"))
        trans_stat = str(row.get("translation_status", "CLEAN"))
        q_en = str(row["question_en"])
        q_id = str(row["question_id"])
        ref_ans = str(row["reference_answer"]).strip().upper()
        orig_img_path = image_map[qa_id]
        
        # Condition P1: Pure Black Canvas (RGB 0,0,0)
        blank_img_path = PERTURB_DIR / f"blank_{cid}.png"
        if not blank_img_path.exists():
            with Image.open(orig_img_path) as orig_pil:
                blank_pil = Image.new("RGB", orig_pil.size, color=(0, 0, 0))
                blank_pil.save(blank_img_path)
                
        # Time sample evaluation with CUDA synchronization
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t_sample_start = time.perf_counter()
        
        # 1. Condition EN-P0 (original image)
        prompt_en = PROMPT_EN_TEMPLATE.format(q=q_en)
        res_en = score_condition_with_kv_cache(model, processor, orig_img_path, prompt_en, CANDIDATES_EN, TOKEN_SETS_EN, target_dev)
        
        # 2. Condition ID-P0 (original image)
        prompt_id = PROMPT_ID_TEMPLATE.format(q=q_id)
        res_id_orig = score_condition_with_kv_cache(model, processor, orig_img_path, prompt_id, CANDIDATES_ID, TOKEN_SETS_ID, target_dev)
        
        # 3. Condition ID-P1 (blank image)
        res_id_blank = score_condition_with_kv_cache(model, processor, blank_img_path, prompt_id, CANDIDATES_ID, TOKEN_SETS_ID, target_dev)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed_sec = float(time.perf_counter() - t_sample_start)
        
        # rev05 Sequence Track Metrics
        corr_en_seq = int(res_en["pred_seq"] == ref_ans)
        corr_id_seq = int(res_id_orig["pred_seq"] == ref_ans)
        flip_seq = int(res_en["pred_seq"] != res_id_orig["pred_seq"])
        joint_corr_seq = int(corr_en_seq == 1 and corr_id_seq == 1)
        p_blank_pred_seq = res_id_blank["p_aff_seq"] if res_id_orig["pred_seq"] == "YES" else res_id_blank["p_neg_seq"]
        delta_p_dir_seq = float(res_id_orig["conf_seq"] - p_blank_pred_seq)
        delta_p_abs_seq = float(abs(res_id_orig["conf_seq"] - p_blank_pred_seq))
        score_m1_seq = float(res_id_orig["conf_seq"])
        score_m2_seq = float(res_id_orig["conf_seq"] * (1.0 - flip_seq))
        score_m_dir_seq = float(res_id_orig["conf_seq"] * max(0.0, delta_p_dir_seq))
        score_m_norm_seq = float(res_id_orig["conf_seq"] * (1.0 - flip_seq) * max(0.0, delta_p_dir_seq))
        score_m3_old_seq = float(res_id_orig["conf_seq"] * (1.0 - flip_seq) * delta_p_abs_seq)
        
        # rev05 Length-Normalized Track Metrics
        corr_en_len = int(res_en["pred_len"] == ref_ans)
        corr_id_len = int(res_id_orig["pred_len"] == ref_ans)
        flip_len = int(res_en["pred_len"] != res_id_orig["pred_len"])
        joint_corr_len = int(corr_en_len == 1 and corr_id_len == 1)
        p_blank_pred_len = res_id_blank["p_aff_len"] if res_id_orig["pred_len"] == "YES" else res_id_blank["p_neg_len"]
        delta_p_dir_len = float(res_id_orig["conf_len"] - p_blank_pred_len)
        delta_p_abs_len = float(abs(res_id_orig["conf_len"] - p_blank_pred_len))
        score_m1_len = float(res_id_orig["conf_len"])
        score_m2_len = float(res_id_orig["conf_len"] * (1.0 - flip_len))
        score_m_dir_len = float(res_id_orig["conf_len"] * max(0.0, delta_p_dir_len))
        score_m_norm_len = float(res_id_orig["conf_len"] * (1.0 - flip_len) * max(0.0, delta_p_dir_len))
        
        # rev06 Multi-Token Track Metrics
        corr_en_v6 = int(res_en["pred_v6"] == ref_ans)
        corr_id_v6 = int(res_id_orig["pred_v6"] == ref_ans)
        flip_v6 = int(res_en["pred_v6"] != res_id_orig["pred_v6"])
        joint_corr_v6 = int(corr_en_v6 == 1 and corr_id_v6 == 1)
        p_blank_pred_v6 = res_id_blank["p_aff_v6"] if res_id_orig["pred_v6"] == "YES" else res_id_blank["p_neg_v6"]
        delta_p_dir_v6 = float(res_id_orig["conf_v6"] - p_blank_pred_v6)
        delta_p_abs_v6 = float(abs(res_id_orig["conf_v6"] - p_blank_pred_v6))
        score_m1_v6 = float(res_id_orig["conf_v6"])
        score_m2_v6 = float(res_id_orig["conf_v6"] * (1.0 - flip_v6))
        score_m_dir_v6 = float(res_id_orig["conf_v6"] * max(0.0, delta_p_dir_v6))
        score_m_norm_v6 = float(res_id_orig["conf_v6"] * (1.0 - flip_v6) * max(0.0, delta_p_dir_v6))
        score_m3_old_v6 = float(res_id_orig["conf_v6"] * (1.0 - flip_v6) * delta_p_abs_v6)
        
        new_records.append({
            "qa_id": qa_id,
            "canonical_image_id": cid,
            "image_id": img_id,
            "split": split_val,
            "translation_status": trans_stat,
            "is_yes_no": bool(row.get("is_yes_no", True)),
            "in_rev03_benchmark": bool(row.get("in_rev03_benchmark", False)),
            "is_eligible_primary": bool(row.get("is_eligible_primary", False)),
            "model_id": MODEL_NAME,
            "device": str(target_dev),
            "gpu_name": str(GPU_NAME),
            "compute_dtype": str(COMPUTE_DTYPE).replace("torch.", ""),
            "elapsed_time_sec": elapsed_sec,
            "question_en": q_en,
            "question_id": q_id,
            "reference_answer": ref_ans,
            
            # Sequence Track Columns (Recomputed via KV-Cache)
            "pred_en_seq": res_en["pred_seq"],
            "conf_en_seq": res_en["conf_seq"],
            "correct_en_seq": corr_en_seq,
            "p_en_yes_seq": res_en["p_aff_seq"],
            "p_en_no_seq": res_en["p_neg_seq"],
            "pred_id_seq": res_id_orig["pred_seq"],
            "conf_id_seq": res_id_orig["conf_seq"],
            "correct_id_seq": corr_id_seq,
            "p_orig_ya_seq": res_id_orig["p_aff_seq"],
            "p_orig_tidak_seq": res_id_orig["p_neg_seq"],
            "flip_seq": flip_seq,
            "joint_correct_seq": joint_corr_seq,
            "pred_blank_id_seq": res_id_blank["pred_seq"],
            "p_blank_ya_seq": res_id_blank["p_aff_seq"],
            "p_blank_tidak_seq": res_id_blank["p_neg_seq"],
            "p_blank_pred_seq": p_blank_pred_seq,
            "delta_p_dir_seq": delta_p_dir_seq,
            "delta_p_abs_seq": delta_p_abs_seq,
            "score_m1_seq": score_m1_seq,
            "score_m2_seq": score_m2_seq,
            "score_m_dir_seq": score_m_dir_seq,
            "score_m_norm_seq": score_m_norm_seq,
            "score_m3_old_seq": score_m3_old_seq,
            
            # Length-Normalized Track Columns
            "pred_en_len": res_en["pred_len"],
            "conf_en_len": res_en["conf_len"],
            "correct_en_len": corr_en_len,
            "p_en_yes_len": res_en["p_aff_len"],
            "p_en_no_len": res_en["p_neg_len"],
            "pred_id_len": res_id_orig["pred_len"],
            "conf_id_len": res_id_orig["conf_len"],
            "correct_id_len": corr_id_len,
            "p_orig_ya_len": res_id_orig["p_aff_len"],
            "p_orig_tidak_len": res_id_orig["p_neg_len"],
            "flip_len": flip_len,
            "joint_correct_len": joint_corr_len,
            "pred_blank_id_len": res_id_blank["pred_len"],
            "p_blank_ya_len": res_id_blank["p_aff_len"],
            "p_blank_tidak_len": res_id_blank["p_neg_len"],
            "p_blank_pred_len": p_blank_pred_len,
            "delta_p_dir_len": delta_p_dir_len,
            "delta_p_abs_len": delta_p_abs_len,
            "score_m1_len": score_m1_len,
            "score_m2_len": score_m2_len,
            "score_m_dir_len": score_m_dir_len,
            "score_m_norm_len": score_m_norm_len,
            
            # rev06 Multi-Token Track Columns
            "pred_en_v6": res_en["pred_v6"],
            "conf_en_v6": res_en["conf_v6"],
            "correct_en_v6": corr_en_v6,
            "p_en_yes_v6": res_en["p_aff_v6"],
            "p_en_no_v6": res_en["p_neg_v6"],
            "pred_id_v6": res_id_orig["pred_v6"],
            "conf_id_v6": res_id_orig["conf_v6"],
            "correct_id_v6": corr_id_v6,
            "p_orig_ya_v6": res_id_orig["p_aff_v6"],
            "p_orig_tidak_v6": res_id_orig["p_neg_v6"],
            "flip_v6": flip_v6,
            "joint_correct_v6": joint_corr_v6,
            "pred_blank_id_v6": res_id_blank["pred_v6"],
            "p_blank_ya_v6": res_id_blank["p_aff_v6"],
            "p_blank_tidak_v6": res_id_blank["p_neg_v6"],
            "p_blank_pred_v6": p_blank_pred_v6,
            "delta_p_dir_v6": delta_p_dir_v6,
            "delta_p_abs_v6": delta_p_abs_v6,
            "score_m1_v6": score_m1_v6,
            "score_m2_v6": score_m2_v6,
            "score_m_dir_v6": score_m_dir_v6,
            "score_m_norm_v6": score_m_norm_v6,
            "score_m3_old_v6": score_m3_old_v6,
        })
        
        if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(df_queue):
            df_final = atomic_save(new_records, checkpoint_dest, df_base=None if PILOT_MODE else df_existing)
            
    t_elapsed = time.time() - t_start
    print(f"✅ Finished inference on {len(df_queue)} samples in {t_elapsed:.2f}s.")

# ==============================================================================


In [ ]:
# ==============================================================================
# STAGE 8C: PILOT PREFLIGHT EVALUATION & HARD GATE DECISION
# ==============================================================================
if PILOT_MODE:
    if len(new_records) == 0 and checkpoint_dest.exists():
        df_pilot = pd.read_csv(checkpoint_dest)
    else:
        df_pilot = pd.DataFrame(new_records)
    print("\n" + "=" * 90)
    print(f"🔬 PILOT EVALUATION REPORT: {MODEL_NAME_SHORT} (N={len(df_pilot)}, GPU: {GPU_NAME}, Dtype: {COMPUTE_DTYPE})")
    print("=" * 90)
    
    # Discover and load stored rev05 checkpoint for ground-truth comparison
    df_stored_rev05 = None
    stored_candidates = [
        Path("artifacts/experiments/decisive_validation/rev05/predictions") / ("qwen" if "qwen" in MODEL_SLUG.lower() and "huatuo" not in MODEL_SLUG.lower() else "huatuo") / f"live_validation_checkpoint_{MODEL_SLUG}.csv",
        RESULTS_ROOT.parent / "rev05" / MODEL_SLUG / f"live_validation_checkpoint_{MODEL_SLUG}.csv",
        RESULTS_ROOT.parent / "rev05" / f"live_validation_checkpoint_{MODEL_SLUG}.csv",
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.rglob("*.csv"):
            if "checkpoint" in p.name.lower() and (MODEL_SLUG.lower() in p.name.lower() or ("qwen" in p.name.lower() if "qwen" in MODEL_SLUG.lower() and "huatuo" not in MODEL_SLUG.lower() else "huatuo" in p.name.lower())):
                stored_candidates.insert(0, p)
                
    for p in stored_candidates:
        if p.exists() and p.is_file():
            cand_df = pd.read_csv(p)
            if len(cand_df) >= 799 and "score_m1_seq" in cand_df.columns:
                df_stored_rev05 = cand_df
                print(f"📁 Loaded stored rev05 checkpoint for ground-truth comparison: {p}")
                break
                
    if df_stored_rev05 is None:
        raise RuntimeError("Stored rev05 checkpoint not found in /kaggle/input! Cannot verify Gate (a) equivalence.")
    else:
        # Merge on qa_id to compare exact same samples
        pilot_merged = df_pilot.merge(
            df_stored_rev05[["qa_id", "pred_en_seq", "p_en_yes_seq", "pred_id_seq", "p_orig_ya_seq", "p_blank_ya_seq", "delta_p_dir_seq"]],
            on="qa_id",
            suffixes=("", "_stored")
        )
        
        # Derive ID-P1 (Blank) predictions from p_blank_ya_seq >= 0.5
        pilot_merged["pred_blank_seq_stored"] = np.where(pilot_merged["p_blank_ya_seq_stored"] >= 0.5, "YES", "NO")
        pilot_merged["pred_blank_seq"] = np.where(pilot_merged["p_blank_ya_seq"] >= 0.5, "YES", "NO")
        
        # Match flags per condition
        match_en = pilot_merged["pred_en_seq"] == pilot_merged["pred_en_seq_stored"]
        match_id_p0 = pilot_merged["pred_id_seq"] == pilot_merged["pred_id_seq_stored"]
        match_id_p1 = pilot_merged["pred_blank_seq"] == pilot_merged["pred_blank_seq_stored"]
        
        # 1. PER-PREDIKSI (Primary Metric)
        # 3 Conditions: EN-P0, ID-P0, ID-P1 (N_pred = 40 * 3 = 120)
        all_pred_matches_3cond = match_en.tolist() + match_id_p0.tolist() + match_id_p1.tolist()
        pred_match_rate_3cond = float(np.mean(all_pred_matches_3cond) * 100.0)
        n_mismatches_3cond = int(len(all_pred_matches_3cond) - sum(all_pred_matches_3cond))
        
        # 2 Conditions: EN-P0, ID-P0 (N_pred = 40 * 2 = 80)
        all_pred_matches_2cond = match_en.tolist() + match_id_p0.tolist()
        pred_match_rate_2cond = float(np.mean(all_pred_matches_2cond) * 100.0)
        n_mismatches_2cond = int(len(all_pred_matches_2cond) - sum(all_pred_matches_2cond))
        
        # 2. PER-SAMPEL (Strict Conjunctive Metric)
        # Sample passes iff ALL its conditions match (N_sample = 40)
        sample_all_match = match_en & match_id_p0 & match_id_p1
        sample_match_rate = float(sample_all_match.mean() * 100.0)
        n_sample_mismatches = int((~sample_all_match).sum())
        
        # Probability differences
        dp_en = np.abs(pilot_merged["p_en_yes_seq"] - pilot_merged["p_en_yes_seq_stored"])
        dp_id_p0 = np.abs(pilot_merged["p_orig_ya_seq"] - pilot_merged["p_orig_ya_seq_stored"])
        dp_id_p1 = np.abs(pilot_merged["p_blank_ya_seq"] - pilot_merged["p_blank_ya_seq_stored"])
        d_deltap = np.abs(pilot_merged["delta_p_dir_seq"] - pilot_merged["delta_p_dir_seq_stored"])
        
        all_dp = np.concatenate([dp_en.values, dp_id_p0.values, dp_id_p1.values])
        max_dp = float(np.max(all_dp))
        mean_dp_orig = float(np.mean(dp_id_p0))
        mean_d_deltap = float(np.mean(d_deltap))
        nan_inf_count = int(np.isnan(all_dp).sum() + np.isinf(all_dp).sum())
        
        # AUDIT KETIDAKCOCOKAN LABEL & MARGIN |p - 0.5|
        mismatch_records = []
        for _, row in pilot_merged.iterrows():
            qid = row["qa_id"]
            if row["pred_en_seq"] != row["pred_en_seq_stored"]:
                p_st = row["p_en_yes_seq_stored"]
                p_pil = row["p_en_yes_seq"]
                mismatch_records.append({
                    "qa_id": qid, "condition": "EN-P0",
                    "stored_pred": row["pred_en_seq_stored"], "pilot_pred": row["pred_en_seq"],
                    "stored_p": round(float(p_st), 5), "pilot_p": round(float(p_pil), 5),
                    "abs_dp": round(float(abs(p_pil - p_st)), 5),
                    "stored_|p-0.5|": round(float(abs(p_st - 0.5)), 5),
                    "pilot_|p-0.5|": round(float(abs(p_pil - 0.5)), 5)
                })
            if row["pred_id_seq"] != row["pred_id_seq_stored"]:
                p_st = row["p_orig_ya_seq_stored"]
                p_pil = row["p_orig_ya_seq"]
                mismatch_records.append({
                    "qa_id": qid, "condition": "ID-P0",
                    "stored_pred": row["pred_id_seq_stored"], "pilot_pred": row["pred_id_seq"],
                    "stored_p": round(float(p_st), 5), "pilot_p": round(float(p_pil), 5),
                    "abs_dp": round(float(abs(p_pil - p_st)), 5),
                    "stored_|p-0.5|": round(float(abs(p_st - 0.5)), 5),
                    "pilot_|p-0.5|": round(float(abs(p_pil - 0.5)), 5)
                })
            if row["pred_blank_seq"] != row["pred_blank_seq_stored"]:
                p_st = row["p_blank_ya_seq_stored"]
                p_pil = row["p_blank_ya_seq"]
                mismatch_records.append({
                    "qa_id": qid, "condition": "ID-P1",
                    "stored_pred": row["pred_blank_seq_stored"], "pilot_pred": row["pred_blank_seq"],
                    "stored_p": round(float(p_st), 5), "pilot_p": round(float(p_pil), 5),
                    "abs_dp": round(float(abs(p_pil - p_st)), 5),
                    "stored_|p-0.5|": round(float(abs(p_st - 0.5)), 5),
                    "pilot_|p-0.5|": round(float(abs(p_pil - 0.5)), 5)
                })

        # Gate decision: (i) >=98% per-prediksi, (ii) setiap mismatch |p-0.5| < 0.02 pada KEDUA nilai, (iii) mean |dp| < 0.02, 0 NaN/Inf
        all_mismatches_marginal = all(m["stored_|p-0.5|"] < 0.02 and m["pilot_|p-0.5|"] < 0.02 for m in mismatch_records)
        gate_passed = bool((pred_match_rate_3cond >= 98.0) and all_mismatches_marginal and (mean_dp_orig < 0.02) and (nan_inf_count == 0))

        
        print("\n" + "-" * 85)
        print("🔍 (a) GATE EKUIVALENSI NUMERIK (Recomputed via KV-Cache vs Stored rev05)")
        print("-" * 85)
        print("  [DEFINISI SATUAN AMBANG 98%]")
        print(f"  • PER-PREDIKSI (3 Kondisi: EN-P0, ID-P0, ID-P1 | N_pred=120):")
        print(f"    - Match Rate:            {pred_match_rate_3cond:.2f}% (Ambang: >= 98.0%) -> {'[PASS]' if pred_match_rate_3cond >= 98.0 else '[FAIL]'}")
        print(f"    - Jumlah Mismatch:       {n_mismatches_3cond} / 120 prediksi")
        print(f"  • PER-PREDIKSI (2 Kondisi: EN-P0, ID-P0 | N_pred=80):")
        print(f"    - Match Rate:            {pred_match_rate_2cond:.2f}%")
        print(f"    - Jumlah Mismatch:       {n_mismatches_2cond} / 80 prediksi")
        print(f"  • PER-SAMPEL (Konjungtif: Ketiga kondisi harus cocok | N_sample=40):")
        print(f"    - Match Rate:            {sample_match_rate:.2f}% ({40 - n_sample_mismatches}/40)")
        print(f"    - Sampel Tidak Cocok:    {n_sample_mismatches} sampel (Catatan: 1 sampel mismatch = 97.5%, sehingga >=98% mensyaratkan 0 mismatch)")
        print(f"  • Mean |dp_orig| (ID-P0):   {mean_dp_orig:.6f} (Ambang: < 0.02) -> {'[PASS]' if mean_dp_orig < 0.02 else '[FAIL]'}")
        print(f"  • Mean |dp| (EN-P0):        {float(np.mean(dp_en)):.6f}")
        print(f"  • Mean |dp| (ID-P1 Blank):  {float(np.mean(dp_id_p1)):.6f}")
        print(f"  • Max |dp| (Semua Kondisi): {max_dp:.6f}")
        print(f"  • Mean |d delta_p_dir|:     {mean_d_deltap:.6f}")
        print(f"  • NaN / Inf Count:          {nan_inf_count} (Ambang: == 0) -> {'[PASS]' if nan_inf_count == 0 else '[FAIL]'}")
        print(f"  • Average Elapsed Time:     {pilot_merged['elapsed_time_sec'].mean():.3f} s/sampel (GPU Sync)")
        
        print("\n  [RINCIAN KETIDAKCOCOKAN LABEL & MARGIN KEPUTUSAN |p - 0.5|]")
        if len(mismatch_records) == 0:
            print("  ✅ 0 KETIDAKCOCOKAN: Seluruh prediksi label pada seluruh kondisi cocok sempurna (100%)!")
        else:
            print(f"  ⚠️ Ditemukan {len(mismatch_records)} ketidakcocokan label:")
            df_mismatches = pd.DataFrame(mismatch_records)
            display(df_mismatches)
            print("  Catatan: Nilai |p - 0.5| sangat kecil (< 0.01) menunjukkan label flip akibat numerik jitter di sekitar ambang batas.")
            
        print("-" * 85)
        # JALUR KEPUTUSAN DIAGNOSTIK RESMI
        if gate_passed:
            print("✅ KESIMPULAN: GATE (a) LOLOS! KV-cache ekuivalen secara numerik dengan rev05.")
            print("   Langkah berikutnya: Ubah PILOT_MODE = False di Sel 8 untuk memulai inferensi full run (1.133 sampel).")
        else:
            print("❌ KESIMPULAN: GATE (a) GAGAL! Ekuivalensi numerik terhadap rev05 tidak tercapai.")
            print("   JALUR KEPUTUSAN DIAGNOSTIK:")
            print("   1. Jika pengujian ini memakai bfloat16, ubah COMPUTE_DTYPE = torch.float16 di Sel 3,")
            print("      lalu jalankan ulang Sel 8 untuk menguji apakah perbedaan berasal dari emulasi bfloat16.")
            print("   2. Jika bfloat16 dan float16 SAMA-SAMA GAGAL:")
            print("      🚨 STOP: jangan jalankan full run; periksa KV-cache/MRoPE/versi library!")
        print("=" * 85)

    # (b) DESKRIPTIF v6 vs seq: Pola Prediksi Tanpa Ambang Lolos/Gagal
    print("\n" + "-" * 75)
    print("📊 (b) DESKRIPTIF rev06 (Multi-Token LogSumExp) vs rev05 (Sequence) [Tanpa Ambang Lolos/Gagal]")
    print("-" * 75)
    
    yes_rate_en_seq = (df_pilot["pred_en_seq"] == "YES").mean() * 100.0
    yes_rate_en_v6 = (df_pilot["pred_en_v6"] == "YES").mean() * 100.0
    
    tp_en_seq = ((df_pilot["pred_en_seq"] == "YES") & (df_pilot["reference_answer"] == "YES")).sum()
    fn_en_seq = ((df_pilot["pred_en_seq"] == "NO") & (df_pilot["reference_answer"] == "YES")).sum()
    tn_en_seq = ((df_pilot["pred_en_seq"] == "NO") & (df_pilot["reference_answer"] == "NO")).sum()
    fp_en_seq = ((df_pilot["pred_en_seq"] == "YES") & (df_pilot["reference_answer"] == "NO")).sum()
    tpr_en_seq = (tp_en_seq / (tp_en_seq + fn_en_seq) * 100.0) if (tp_en_seq + fn_en_seq) > 0 else 0.0
    tnr_en_seq = (tn_en_seq / (tn_en_seq + fp_en_seq) * 100.0) if (tn_en_seq + fp_en_seq) > 0 else 0.0
    
    tp_en_v6 = ((df_pilot["pred_en_v6"] == "YES") & (df_pilot["reference_answer"] == "YES")).sum()
    fn_en_v6 = ((df_pilot["pred_en_v6"] == "NO") & (df_pilot["reference_answer"] == "YES")).sum()
    tn_en_v6 = ((df_pilot["pred_en_v6"] == "NO") & (df_pilot["reference_answer"] == "NO")).sum()
    fp_en_v6 = ((df_pilot["pred_en_v6"] == "YES") & (df_pilot["reference_answer"] == "NO")).sum()
    tpr_en_v6 = (tp_en_v6 / (tp_en_v6 + fn_en_v6) * 100.0) if (tp_en_v6 + fn_en_v6) > 0 else 0.0
    tnr_en_v6 = (tn_en_v6 / (tn_en_v6 + fp_en_v6) * 100.0) if (tn_en_v6 + fp_en_v6) > 0 else 0.0
    
    yes_rate_id_seq = (df_pilot["pred_id_seq"] == "YES").mean() * 100.0
    yes_rate_id_v6 = (df_pilot["pred_id_v6"] == "YES").mean() * 100.0
    
    tp_id_seq = ((df_pilot["pred_id_seq"] == "YES") & (df_pilot["reference_answer"] == "YES")).sum()
    fn_id_seq = ((df_pilot["pred_id_seq"] == "NO") & (df_pilot["reference_answer"] == "YES")).sum()
    tn_id_seq = ((df_pilot["pred_id_seq"] == "NO") & (df_pilot["reference_answer"] == "NO")).sum()
    fp_id_seq = ((df_pilot["pred_id_seq"] == "YES") & (df_pilot["reference_answer"] == "NO")).sum()
    tpr_id_seq = (tp_id_seq / (tp_id_seq + fn_id_seq) * 100.0) if (tp_id_seq + fn_id_seq) > 0 else 0.0
    tnr_id_seq = (tn_id_seq / (tn_id_seq + fp_id_seq) * 100.0) if (tn_id_seq + fp_id_seq) > 0 else 0.0
    
    tp_id_v6 = ((df_pilot["pred_id_v6"] == "YES") & (df_pilot["reference_answer"] == "YES")).sum()
    fn_id_v6 = ((df_pilot["pred_id_v6"] == "NO") & (df_pilot["reference_answer"] == "YES")).sum()
    tn_id_v6 = ((df_pilot["pred_id_v6"] == "NO") & (df_pilot["reference_answer"] == "NO")).sum()
    fp_id_v6 = ((df_pilot["pred_id_v6"] == "YES") & (df_pilot["reference_answer"] == "NO")).sum()
    tpr_id_v6 = (tp_id_v6 / (tp_id_v6 + fn_id_v6) * 100.0) if (tp_id_v6 + fn_id_v6) > 0 else 0.0
    tnr_id_v6 = (tn_id_v6 / (tn_id_v6 + fp_id_v6) * 100.0) if (tn_id_v6 + fp_id_v6) > 0 else 0.0
    
    flip_rate_seq = (df_pilot["pred_en_seq"] != df_pilot["pred_id_seq"]).mean() * 100.0
    flip_rate_v6 = (df_pilot["pred_en_v6"] != df_pilot["pred_id_v6"]).mean() * 100.0
    
    desc_df = pd.DataFrame([
        {"Metrik": "Predicted-YES Rate EN (%)", "rev05 (Sequence)": f"{yes_rate_en_seq:.2f}%", "rev06 (LogSumExp v6)": f"{yes_rate_en_v6:.2f}%"},
        {"Metrik": "TPR EN (Sensitivitas %)",   "rev05 (Sequence)": f"{tpr_en_seq:.2f}%",      "rev06 (LogSumExp v6)": f"{tpr_en_v6:.2f}%"},
        {"Metrik": "TNR EN (Spesifisitas %)",   "rev05 (Sequence)": f"{tnr_en_seq:.2f}%",      "rev06 (LogSumExp v6)": f"{tnr_en_v6:.2f}%"},
        {"Metrik": "Predicted-YES Rate ID (%)", "rev05 (Sequence)": f"{yes_rate_id_seq:.2f}%", "rev06 (LogSumExp v6)": f"{yes_rate_id_v6:.2f}%"},
        {"Metrik": "TPR ID (Sensitivitas %)",   "rev05 (Sequence)": f"{tpr_id_seq:.2f}%",      "rev06 (LogSumExp v6)": f"{tpr_id_v6:.2f}%"},
        {"Metrik": "TNR ID (Spesifisitas %)",   "rev05 (Sequence)": f"{tnr_id_seq:.2f}%",      "rev06 (LogSumExp v6)": f"{tnr_id_v6:.2f}%"},
        {"Metrik": "Flip Rate EN!=ID (%)",      "rev05 (Sequence)": f"{flip_rate_seq:.2f}%",   "rev06 (LogSumExp v6)": f"{flip_rate_v6:.2f}%"}
    ])
    display(desc_df)
    print("=" * 90)


In [ ]:
# ==============================================================================
# STAGE 9: POST-INFERENCE CHECKPOINT INTEGRITY AUDIT
# ==============================================================================
print("=" * 70)
print("🔍 POST-INFERENCE CHECKPOINT INTEGRITY AUDIT")
print("=" * 70)

if PILOT_MODE:
    print(f"ℹ️ PILOT MODE ACTIVE: Skipping full-run 1,133-row check. Verifying {len(df_final)} pilot samples...")
    assert len(df_final) == PILOT_SAMPLE_COUNT, f"Expected {PILOT_SAMPLE_COUNT} pilot rows, found {len(df_final)}"
    print(f"✅ Pilot Preflight Row Count: {len(df_final)} / {PILOT_SAMPLE_COUNT} (100% COMPLETE)")
else:
    assert len(df_final) == 1133, f"Incomplete checkpoint! Expected 1133 rows, found {len(df_final)}"
    assert df_final["qa_id"].nunique() == 1133, "Duplicate QA IDs detected in checkpoint!"
    assert set(df_final["qa_id"]) == set(df_eval_cohort["qa_id"]), "QA ID mismatch against evaluation cohort!"
    print("✅ Full Run Row Count: 1,133 / 1,133 (100% COMPLETE)")

# Non-finite score check
score_cols = [
    "score_m1_seq", "score_m2_seq", "score_m_dir_seq", "score_m_norm_seq", "score_m3_old_seq",
    "delta_p_dir_seq", "score_m1_len", "score_m_norm_len"
]
for col in score_cols:
    assert np.isfinite(df_final[col]).all(), f"Non-finite values detected in {col}!"

print("✅ Duplicate QA IDs: 0")
print("✅ Non-finite Values: 0")
print(f"✅ Accuracy ID (Seq): {df_final['correct_id_seq'].mean()*100:.2f}%")
print(f"✅ Accuracy EN (Seq): {df_final['correct_en_seq'].mean()*100:.2f}%")
print(f"✅ Flip Rate (Seq)  : {df_final['flip_seq'].mean()*100:.2f}%")
print("=" * 70)


In [ ]:
# ==============================================================================
# STAGE 4: STATISTICAL EVALUATION ENGINE (ZERO-GPU RIEMANN AURC & BOOTSTRAP)
# ==============================================================================

def calculate_discrete_aurc(correct_array, score_array):
    N = len(correct_array)
    if N == 0:
        return 0.0, np.array([]), np.array([])
    sort_idx = np.argsort(-score_array, kind="mergesort")
    sorted_corr = correct_array[sort_idx]
    k_vals = np.arange(1, N + 1)
    coverages = k_vals / float(N)
    selective_risks = 1.0 - (np.cumsum(sorted_corr) / k_vals.astype(float))
    discrete_aurc = float(np.mean(selective_risks))
    return discrete_aurc, coverages, selective_risks

def get_risk_at_coverage(cov_arr, risk_arr, target_cov):
    idx = int((np.abs(cov_arr - target_cov)).argmin())
    return float(risk_arr[idx])

def paired_cluster_permutation_test(df, score_col_m, score_col_base="score_m1_seq", 
                                    corr_col="correct_id_seq", cluster_col="canonical_image_id", 
                                    n_perm=1000, seed=42):
    df = df.reset_index(drop=True)
    rng = np.random.RandomState(seed)
    
    cluster_vals = df[cluster_col].to_numpy()
    clusters = np.unique(cluster_vals)
    n_clusters = len(clusters)
    cluster_to_pos = {c: np.where(cluster_vals == c)[0] for c in clusters}
    
    y = df[corr_col].to_numpy(dtype=float)
    s_m = df[score_col_m].to_numpy(dtype=float)
    s_base = df[score_col_base].to_numpy(dtype=float)
    
    obs_m, _, _ = calculate_discrete_aurc(y, s_m)
    obs_base, _, _ = calculate_discrete_aurc(y, s_base)
    obs_diff = abs(obs_m - obs_base)
    
    if obs_diff == 0.0:
        return 1.0
        
    extreme = 0
    for _ in range(n_perm):
        sm_p = s_m.copy()
        sb_p = s_base.copy()
        flips = rng.choice([-1, 1], size=n_clusters)
        for cid, f in zip(clusters, flips):
            if f == -1:
                idx = cluster_to_pos[cid]
                sm_p[idx], sb_p[idx] = s_base[idx], s_m[idx]
        pm, _, _ = calculate_discrete_aurc(y, sm_p)
        pb, _, _ = calculate_discrete_aurc(y, sb_p)
        if abs(pm - pb) >= obs_diff:
            extreme += 1
            
    return float((extreme + 1) / (n_perm + 1))

def paired_cluster_bootstrap(df, score_cols, corr_col="correct_id_seq", cluster_col="canonical_image_id", n_boot=1000, seed=42):
    df = df.reset_index(drop=True)
    rng = np.random.RandomState(seed)
    
    cluster_vals = df[cluster_col].to_numpy()
    clusters = np.unique(cluster_vals)
    n_clusters = len(clusters)
    cluster_to_pos = {c: np.where(cluster_vals == c)[0] for c in clusters}
    
    y = df[corr_col].to_numpy(dtype=float)
    scores_dict = {col: df[col].to_numpy(dtype=float) for col in score_cols}
    base_col = score_cols[0]
    
    boot_aurcs = {col: [] for col in score_cols}
    boot_deltas = {col: [] for col in score_cols if col != base_col}
    
    for _ in range(n_boot):
        sampled_c = rng.choice(clusters, size=n_clusters, replace=True)
        samp_idx = np.concatenate([cluster_to_pos[c] for c in sampled_c])
        y_b = y[samp_idx]
        
        aurc_b = {}
        for col in score_cols:
            a, _, _ = calculate_discrete_aurc(y_b, scores_dict[col][samp_idx])
            boot_aurcs[col].append(a)
            aurc_b[col] = a
            
        for col in boot_deltas:
            boot_deltas[col].append(aurc_b[col] - aurc_b[base_col])
            
    summary = {}
    for col in score_cols:
        vals = np.array(boot_aurcs[col])
        summary[col] = {
            "mean": float(np.mean(vals)),
            "ci_lower": float(np.percentile(vals, 2.5)),
            "ci_upper": float(np.percentile(vals, 97.5))
        }
    diff_summary = {}
    for col in boot_deltas:
        d_vals = np.array(boot_deltas[col])
        diff_summary[col] = {
            "delta_mean": float(np.mean(d_vals)),
            "delta_ci_lower": float(np.percentile(d_vals, 2.5)),
            "delta_ci_upper": float(np.percentile(d_vals, 97.5))
        }
    return summary, diff_summary

def apply_holm_bonferroni(p_vals_dict):
    sorted_pairs = sorted(p_vals_dict.items(), key=lambda x: x[1])
    m = len(sorted_pairs)
    adj = {}
    run_max = 0.0
    for r, (k, p) in enumerate(sorted_pairs, 1):
        mult = m - r + 1
        adj_p = min(1.0, p * mult)
        run_max = max(run_max, adj_p)
        adj[k] = min(1.0, run_max)
    return adj

print("✅ Statistical Testing Engine (Positional Indices) initialized.")

In [ ]:
# ==============================================================================
# STAGE 4B: AUTOMATED PREFLIGHT VERIFICATION & INDEX SANITY TEST
# ==============================================================================
print("=" * 70)
print("🧪 RUNNING AUTOMATED PREFLIGHT STATISTICAL ENGINE & INDEX TESTS...")
print("=" * 70)

# 1. Non-standard index test (checks index slicing immunity)
mock_df = pd.DataFrame({
    "canonical_image_id": ["c1", "c1", "c2", "c2", "c3", "c3", "c4"],
    "correct_id_seq": [1, 0, 1, 1, 0, 1, 0],
    "score_m1_seq": [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3],
    "score_m_norm_seq": [0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25]
}, index=[10, 25, 99, 150, 777, 1050, 2024])

p_mock = paired_cluster_permutation_test(mock_df, "score_m_norm_seq", "score_m1_seq", n_perm=50, seed=42)
assert 0.0 <= p_mock <= 1.0, f"Invalid permutation p-value: {p_mock}"
print(f"✅ Test 1: Permutation test with non-standard index [10..2024] -> p = {p_mock:.4f} (No IndexError)")

b_sum_m, b_diff_m = paired_cluster_bootstrap(mock_df, ["score_m1_seq", "score_m_norm_seq"], n_boot=50, seed=42)
assert b_diff_m["score_m_norm_seq"]["delta_ci_lower"] <= b_diff_m["score_m_norm_seq"]["delta_ci_upper"]
print("✅ Test 2: Cluster bootstrap with non-standard index [10..2024] -> CIs valid (No IndexError)")

# 2. AURC mathematical boundary invariants
y_perf = np.ones(50)
s_test = np.linspace(1.0, 0.0, 50)
a_perf, _, _ = calculate_discrete_aurc(y_perf, s_test)
assert a_perf == 0.0, f"Expected perfect AURC 0.0, got {a_perf}"

y_zero = np.zeros(50)
a_zero, _, _ = calculate_discrete_aurc(y_zero, s_test)
assert a_zero == 1.0, f"Expected zero-accuracy AURC 1.0, got {a_zero}"
print("✅ Test 3: Mathematical boundary invariants (Perfect=0.0, Zero=1.0) verified.")

# 3. Holm-Bonferroni step-down monotonicity
p_dict = {"m2": 0.04, "m_norm": 0.01, "m_dir": 0.03}
adj_p = apply_holm_bonferroni(p_dict)
assert adj_p["m_norm"] <= adj_p["m_dir"] <= adj_p["m2"]
print(f"✅ Test 4: Holm-Bonferroni monotonicity verified: {adj_p}")

print("=" * 70)
print("🎉 ALL PREFLIGHT STATISTICAL & INDEX TESTS PASSED (100% PASS)!")
print("🚀 Proceeding safely to Final Metrics Computation...")
print("=" * 70)

In [ ]:
# ==============================================================================
# STAGE 12: RQ1 BILINGUAL ACCURACY & CROSS-LINGUAL DISAGREEMENT
# ==============================================================================

def evaluate_rq1(df, model_title):
    subsets = [
        ("primary_test", df[(df["split"] == "test") & (df["is_eligible_primary"] == True)]),
        ("primary_dev", df[(df["split"] == "dev") & (df["is_eligible_primary"] == True)]),
        ("historical_test", df[(df["split"] == "test") & (df["in_rev03_benchmark"] == True)])
    ]
    rows = []
    for cohort_name, sub in subsets:
        if len(sub) == 0:
            continue
        acc_en_seq = sub["correct_en_seq"].mean() * 100.0
        acc_id_seq = sub["correct_id_seq"].mean() * 100.0
        gap_seq = acc_en_seq - acc_id_seq
        flip_seq = sub["flip_seq"].mean() * 100.0
        joint_acc_seq = sub["joint_correct_seq"].mean() * 100.0
        
        acc_en_len = sub["correct_en_len"].mean() * 100.0
        acc_id_len = sub["correct_id_len"].mean() * 100.0
        gap_len = acc_en_len - acc_id_len
        flip_len = sub["flip_len"].mean() * 100.0
        joint_acc_len = sub["joint_correct_len"].mean() * 100.0
        
        rows.append({
            "model": model_title,
            "cohort": cohort_name,
            "n_samples": len(sub),
            "acc_en_seq": acc_en_seq,
            "acc_id_seq": acc_id_seq,
            "gap_seq": gap_seq,
            "flip_rate_seq": flip_seq,
            "joint_acc_seq": joint_acc_seq,
            "acc_en_len": acc_en_len,
            "acc_id_len": acc_id_len,
            "gap_len": gap_len,
            "flip_rate_len": flip_len,
            "joint_acc_len": joint_acc_len
        })
    return rows

rq1_data = evaluate_rq1(df_final, MODEL_NAME_SHORT)
df_rq1 = pd.DataFrame(rq1_data)
rq1_csv = RESULTS_ROOT / f"rq1_bilingual_metrics_{MODEL_SLUG}.csv"
df_rq1.to_csv(rq1_csv, index=False)
print("=" * 80)
print(f"📊 RQ1 BILINGUAL METRICS SUMMARY: {MODEL_NAME_SHORT} (PRIMARY TEST N=799)")
print("=" * 80)
display(df_rq1[df_rq1["cohort"] == "primary_test"][["model", "acc_en_seq", "acc_id_seq", "gap_seq", "flip_rate_seq", "joint_acc_seq"]])
print("=" * 80)


In [ ]:
# ==============================================================================
# STAGE 13: RQ2 MULTIMODAL VISUAL INTERVENTION (P0 ORIGINAL VS P1 BLANK)
# ==============================================================================

def evaluate_rq2(df, model_title):
    subsets = [
        ("primary_test", df[(df["split"] == "test") & (df["is_eligible_primary"] == True)]),
        ("historical_test", df[(df["split"] == "test") & (df["in_rev03_benchmark"] == True)])
    ]
    rows = []
    for cohort_name, sub in subsets:
        if len(sub) == 0:
            continue
        mean_delta_seq = sub["delta_p_dir_seq"].mean()
        prop_pos_seq = (sub["delta_p_dir_seq"] > 0).mean() * 100.0
        prop_neg_seq = (sub["delta_p_dir_seq"] < 0).mean() * 100.0
        
        mean_delta_len = sub["delta_p_dir_len"].mean()
        prop_pos_len = (sub["delta_p_dir_len"] > 0).mean() * 100.0
        prop_neg_len = (sub["delta_p_dir_len"] < 0).mean() * 100.0
        
        rows.append({
            "model": model_title,
            "cohort": cohort_name,
            "n_samples": len(sub),
            "mean_delta_p_seq": mean_delta_seq,
            "prop_pos_reliance_seq": prop_pos_seq,
            "prop_neg_reliance_seq": prop_neg_seq,
            "mean_delta_p_len": mean_delta_len,
            "prop_pos_reliance_len": prop_pos_len,
            "prop_neg_reliance_len": prop_neg_len
        })
    return rows

rq2_data = evaluate_rq2(df_final, MODEL_NAME_SHORT)
df_rq2 = pd.DataFrame(rq2_data)
rq2_csv = RESULTS_ROOT / f"rq2_visual_intervention_{MODEL_SLUG}.csv"
df_rq2.to_csv(rq2_csv, index=False)
print("=" * 80)
print(f"📊 RQ2 VISUAL INTERVENTION SUMMARY: {MODEL_NAME_SHORT} (PRIMARY TEST N=799)")
print("=" * 80)
display(df_rq2[df_rq2["cohort"] == "primary_test"])
print("=" * 80)


In [ ]:
import matplotlib.pyplot as plt
# ==============================================================================
# STAGE 7: RQ3 SELECTIVE PREDICTION & RIGOROUS STATISTICAL TESTING
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def run_selective_prediction(df_subset, model_title, track_label="primary_test"):
    df_subset = df_subset.reset_index(drop=True)
    n_samples = len(df_subset)
    n_clusters = df_subset["canonical_image_id"].nunique()
    
    # Primary Sequence Track
    score_cols_seq = ["score_m1_seq", "score_m2_seq", "score_m_dir_seq", "score_m_norm_seq", "score_m3_old_seq"]
    boot_sum_seq, boot_diff_seq = paired_cluster_bootstrap(df_subset, score_cols_seq, corr_col="correct_id_seq")
    
    rows = []
    raw_p_seq = {}
    curves = {}
    base_aurc = None
    
    for col in score_cols_seq:
        aurc, cov, risk = calculate_discrete_aurc(df_subset["correct_id_seq"].values, df_subset[col].values)
        curves[col] = (cov, risk)
        if col == "score_m1_seq":
            base_aurc = aurc
            p_val = 1.0
            delta_aurc = 0.0
        else:
            p_val = paired_cluster_permutation_test(df_subset, col, "score_m1_seq", corr_col="correct_id_seq")
            raw_p_seq[col] = p_val
            delta_aurc = aurc - base_aurc
            
        r100 = get_risk_at_coverage(cov, risk, 1.0)
        r90 = get_risk_at_coverage(cov, risk, 0.9)
        r80 = get_risk_at_coverage(cov, risk, 0.8)
        r70 = get_risk_at_coverage(cov, risk, 0.7)
        r50 = get_risk_at_coverage(cov, risk, 0.5)
        
        rows.append({
            "model": model_title, "cohort": track_label, "scoring_track": "sequence_logprob",
            "n_samples": n_samples, "n_clusters": n_clusters, "estimator": col,
            "aurc_mean": aurc, "ci_95_lower": boot_sum_seq[col]["ci_lower"], "ci_95_upper": boot_sum_seq[col]["ci_upper"],
            "delta_aurc": delta_aurc,
            "delta_ci_lower": boot_diff_seq[col]["delta_ci_lower"] if col != "score_m1_seq" else 0.0,
            "delta_ci_upper": boot_diff_seq[col]["delta_ci_upper"] if col != "score_m1_seq" else 0.0,
            "perm_p_raw": p_val,
            "risk_100": r100, "risk_90": r90, "risk_80": r80, "risk_70": r70, "risk_50": r50
        })
        
    adj_p = apply_holm_bonferroni(raw_p_seq)
    for r in rows:
        r["perm_p_holm"] = adj_p.get(r["estimator"], 1.0)
        
    # Plot Risk-Coverage Curve
    safe_m = model_title.replace("/", "_").replace("-", "_")
    plt.figure(figsize=(8, 6), dpi=150)
    for col, (cov, risk) in curves.items():
        aurc_val = [r["aurc_mean"] for r in rows if r["estimator"] == col][0]
        plt.plot(cov, risk, label=f"{col.replace('_seq','')} (AURC={aurc_val:.4f})", lw=2)
    plt.xlabel("Coverage (Proportion of Evaluated Cases)", fontsize=11)
    plt.ylabel("Selective Diagnostic Risk (1 - Accuracy)", fontsize=11)
    plt.title(f"Risk-Coverage Trade-Off: {model_title}\n({track_label.upper()} N={n_samples}, {n_clusters} Clusters)", fontsize=12, fontweight="bold")
    plt.legend(frameon=True, fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f"risk_coverage_{track_label}_{safe_m}.png")
    plt.close()
    
    return pd.DataFrame(rows)



# Evaluate Primary Test Set (N=799, 204 Clusters)
df_test_primary = df_final[(df_final["split"] == "test") & (df_final["is_eligible_primary"] == True)]
df_rq3 = run_selective_prediction(df_test_primary, MODEL_NAME_SHORT, "primary_test")

rq3_csv = RESULTS_ROOT / f"rq3_selective_prediction_{MODEL_SLUG}.csv"
df_rq3.to_csv(rq3_csv, index=False)

print("=" * 95)
print(f"🎯 RQ3 SELECTIVE PREDICTION: {MODEL_NAME_SHORT} (PRIMARY TEST N=799, 204 CLUSTERS)")
print("=" * 95)
cols_show = ["model", "estimator", "aurc_mean", "ci_95_lower", "ci_95_upper", "delta_aurc", "perm_p_holm", "risk_100", "risk_80", "risk_50"]
display(df_rq3[cols_show])
print("=" * 95)


In [ ]:
# ==============================================================================
# STAGE 8: LEARNED RISK ABLATION (D_LANG & DEV SET CALIBRATION)
# ==============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

def evaluate_risk_ablation(df_full, model_title):
    df_dev = df_full[(df_full["split"] == "dev") & (df_full["is_eligible_primary"] == True)].copy().reset_index(drop=True)
    df_test = df_full[(df_full["split"] == "test") & (df_full["is_eligible_primary"] == True)].copy().reset_index(drop=True)
    
    for df in [df_dev, df_test]:
        df["d_lang"] = (df["pred_en_seq"] != df["pred_id_seq"]).astype(int)
        df["error_id"] = (1 - df["correct_id_seq"]).astype(int)
        
    feature_sets = {
        "LR_conf": ["score_m1_seq"],
        "LR_visual": ["score_m1_seq", "delta_p_dir_seq"],
        "LR_full": ["score_m1_seq", "delta_p_dir_seq", "d_lang"]
    }
    
    rows = []
    gkf = GroupKFold(n_splits=5)
    groups_dev = df_dev["canonical_image_id"].values
    y_dev = df_dev["error_id"].values
    
    for mname, feats in feature_sets.items():
        X_dev_raw = df_dev[feats].values
        X_test_raw = df_test[feats].values
        
        # Out-of-fold CV on Dev Set
        oof_risk = np.zeros(len(df_dev))
        for tr_idx, v_idx in gkf.split(X_dev_raw, y_dev, groups=groups_dev):
            scaler_cv = StandardScaler()
            X_tr = scaler_cv.fit_transform(X_dev_raw[tr_idx])
            X_v = scaler_cv.transform(X_dev_raw[v_idx])
            clf = LogisticRegression(C=1.0, random_state=42)
            clf.fit(X_tr, y_dev[tr_idx])
            oof_risk[v_idx] = clf.predict_proba(X_v)[:, 1]
            
        dev_scores = 1.0 - oof_risk
        dev_aurc, _, _ = calculate_discrete_aurc(df_dev["correct_id_seq"].values, dev_scores)
        
        # Train on full Dev Set, evaluate on Test Set
        final_scaler = StandardScaler()
        X_dev_sc = final_scaler.fit_transform(X_dev_raw)
        X_test_sc = final_scaler.transform(X_test_raw)
        final_clf = LogisticRegression(C=1.0, random_state=42)
        final_clf.fit(X_dev_sc, y_dev)
        
        test_risk = final_clf.predict_proba(X_test_sc)[:, 1]
        test_scores = 1.0 - test_risk
        test_aurc, _, _ = calculate_discrete_aurc(df_test["correct_id_seq"].values, test_scores)
        
        rows.append({
            "model": model_title, "ablation_model": mname, "features": ", ".join(feats),
            "dev_aurc_cv": dev_aurc, "test_aurc": test_aurc,
            "coefs": [round(float(c), 4) for c in final_clf.coef_[0]],
            "intercept": round(float(final_clf.intercept_[0]), 4)
        })
    return rows


ablation_data = evaluate_risk_ablation(df_final, MODEL_NAME_SHORT)
df_ablation = pd.DataFrame(ablation_data)
ablation_csv = RESULTS_ROOT / f"risk_ablation_{MODEL_SLUG}.csv"
df_ablation.to_csv(ablation_csv, index=False)

print("=" * 80)
print(f"🔬 LEARNED RISK ABLATION: {MODEL_NAME_SHORT} (DEV N=323 CV -> TEST N=799)")
print("=" * 80)
display(df_ablation)
print("=" * 80)


In [ ]:
# ==============================================================================
# STAGE 16: ARTIFACT PACKAGING & 1-CLICK DOWNLOAD
# ==============================================================================
import zipfile
import shutil
from datetime import datetime, timezone
from IPython.display import FileLink

zip_name = f"{MODEL_SLUG}_complete_validation_results.zip"
zip_path = Path("/kaggle/working") / zip_name if Path("/kaggle/working").exists() else RESULTS_ROOT / zip_name

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(checkpoint_dest, arcname=checkpoint_dest.name)
    if rq1_csv.exists():
        zf.write(rq1_csv, arcname=rq1_csv.name)
    if rq2_csv.exists():
        zf.write(rq2_csv, arcname=rq2_csv.name)
    if rq3_csv.exists():
        zf.write(rq3_csv, arcname=rq3_csv.name)
    if ablation_csv.exists():
        zf.write(ablation_csv, arcname=ablation_csv.name)
    for plot_p in RESULTS_ROOT.glob("*.png"):
        zf.write(plot_p, arcname=plot_p.name)

print("=" * 70)
print(f"📦 EXECUTION & EVALUATION 100% COMPLETE! Archive: {zip_path.name}")
print(f"   Size: {zip_path.stat().st_size / (1024*1024):.2f} MB")
print("=" * 70)
display(FileLink(str(zip_path)))
